# Movie Review Rater with Reasoning LLM

Load movie reviews from the IMDB dataset on HuggingFace, then use a reasoning LLM
to analyze and rate them on a 1-10 scale with a detailed explanation.

**What this project demonstrates:**
- HuggingFace `datasets` library for loading and exploring datasets
- Reasoning LLMs that show their thinking process
- Structured prompting for classification with explanation
- Comparing model reasoning vs ground-truth labels
- Gradio interface for interactive review analysis

Built as part of Module 6 (HuggingFace 2) of the LLM & Agentic AI Bootcamp.

**Note:** This notebook uses OpenRouter to access reasoning models without GPU requirements.

## Setup

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from datasets import load_dataset
import gradio as gr
from IPython.display import display, Markdown

load_dotenv()

# OpenRouter client for accessing reasoning models
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

print("Client ready!")

/Users/eugenionex/Downloads/LLM and Agentic AI Bootcamp Materials/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Client ready!


## Load IMDB Dataset from HuggingFace

The `datasets` library by HuggingFace provides easy access to thousands of datasets.
IMDB contains 50,000 movie reviews labeled as positive or negative.

In [2]:
# Load the IMDB dataset (test split - 25,000 reviews)
dataset = load_dataset("imdb", split="test")

print(f"Dataset size: {len(dataset)} reviews")
print(f"Features: {dataset.features}")
print(f"Labels: 0 = negative, 1 = positive")

Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 475488.60 examples/s]

Dataset size: 25000 reviews
Features: {'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}
Labels: 0 = negative, 1 = positive


In [3]:
# Explore the dataset as a Pandas DataFrame
df = dataset.to_pandas()
print(f"\nLabel distribution:")
print(df["label"].value_counts().rename({0: "Negative", 1: "Positive"}))
print(f"\nAverage review length: {df['text'].str.len().mean():.0f} characters")
print(f"\nSample review (first 200 chars):")
print(df.iloc[0]["text"][:200] + "...")


Label distribution:
label
Negative    12500
Positive    12500
Name: count, dtype: int64

Average review length: 1294 characters

Sample review (first 200 chars):
I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Bab...


## Reasoning Model Configuration

We'll use a reasoning model that shows its thinking process - 
the same concept as DeepSeek-R1 from the module, but accessed via OpenRouter.

In [4]:
# Reasoning model via OpenRouter
REASONING_MODEL = "deepseek/deepseek-r1"

# Fast model for comparison
FAST_MODEL = "openai/gpt-4o-mini"

print(f"Reasoning model: {REASONING_MODEL}")
print(f"Fast model: {FAST_MODEL}")

Reasoning model: deepseek/deepseek-r1
Fast model: openai/gpt-4o-mini


## Review Analysis Prompt

The prompt asks the model to:
1. Rate the movie (not the review) on a 1-10 scale based on the review
2. Classify the sentiment
3. Show its reasoning step by step

In [5]:
ANALYSIS_PROMPT = """
You are an expert film critic and sentiment analyst.

Analyze the following movie review and provide:

1. **Sentiment:** Positive, Negative, or Mixed
2. **Rating:** A score from 1-10 for the movie based on this review
3. **Key Points:** The main praise or criticism mentioned (2-3 bullet points)
4. **Reasoning:** Explain step by step how you arrived at your rating

Format your response as:

**Sentiment:** [Positive/Negative/Mixed]
**Rating:** [X]/10

**Key Points:**
- ...
- ...

**Reasoning:**
[Your step-by-step analysis]
"""

print("Analysis prompt ready!")

Analysis prompt ready!


In [6]:
def analyze_review(review_text, model=REASONING_MODEL):
    """Analyze a movie review using a reasoning LLM."""
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": ANALYSIS_PROMPT},
                {"role": "user", "content": f"Movie Review:\n\n{review_text[:2000]}"}
            ],
            temperature=0.3
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

## Test on Sample Reviews

Let's pick some reviews from the dataset and see how the reasoning model analyzes them.

In [7]:
# Analyze a positive review from the dataset
positive_review = dataset.filter(lambda x: x["label"] == 1)[0]["text"]

print(f"Review (first 300 chars): {positive_review[:300]}...")
print(f"Ground truth: POSITIVE\n")
print("Analyzing with reasoning model...\n")

result = analyze_review(positive_review)
display(Markdown(result))

Filter: 100%|██████████| 25000/25000 [00:00<00:00, 329263.55 examples/s]

Review (first 300 chars): Previous reviewer Claudio Carvalho gave a much better recap of the film's plot details than I could. What I recall mostly is that it was just so beautiful, in every sense - emotionally, visually, editorially - just gorgeous.<br /><br />If you like movies that are wonderful to look at, and also have ...
Ground truth: POSITIVE

Analyzing with reasoning model...



**Sentiment:** Positive  
**Rating:** 8.75/10  

**Key Points:**  
- Praises the film’s beauty in emotional, visual, and editorial execution.  
- Highlights it as an extraordinary, unusual work of art with relevant emotional depth.  
- Notes that its impact is mood-dependent, best appreciated when viewers are in a specific mindset.  

**Reasoning:**  
1. **Sentiment Analysis:** The review uses overwhelmingly positive language (“beautiful,” “gorgeous,” “extraordinary,” “must-see”) to describe the film. Even the critique about its mood-dependent nature is framed as a situational limitation rather than a flaw.  
2. **Rating Calculation:** The reviewer explicitly assigns an 8.75/10, explaining that the score reflects the film’s status as a “mood piece.” They clarify that it could be a 10/10 for viewers in the right emotional state, but averages lower due to its niche appeal.  
3. **Key Points Identification:** The admiration for the film’s aesthetics and emotional resonance dominates the review, alongside acknowledgment of its unconventional artistry. The mood-dependent caveat is noted but does not overshadow the overall praise.  
4. **Final Judgment:** While the score is slightly tempered by the film’s specificity, the overwhelming positivity in language and endorsement (“must-see”) justify the high rating and Positive sentiment.

In [8]:
# Analyze a negative review
negative_review = dataset.filter(lambda x: x["label"] == 0)[0]["text"]

print(f"Review (first 300 chars): {negative_review[:300]}...")
print(f"Ground truth: NEGATIVE\n")
print("Analyzing with reasoning model...\n")

result = analyze_review(negative_review)
display(Markdown(result))

Filter: 100%|██████████| 25000/25000 [00:00<00:00, 381929.51 examples/s]


Review (first 300 chars): I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, C...
Ground truth: NEGATIVE

Analyzing with reasoning model...



**Sentiment:** Negative  
**Rating:** 2/10  

**Key Points:**  
- Criticizes poor production quality (cheap sets, mismatched CGI, silly prosthetics).  
- Laments wooden characters, stilted dialogue, and lack of emotional depth.  
- Highlights incoherent editing and forced references to Gene Roddenberry to mask flaws.  

**Reasoning:**  
1. **Negative Sentiment:** The review opens with the author’s love for sci-fi but quickly pivots to frustration. Phrases like “painfully one-dimensional,” “cheap cardboard sets,” and “dull, cheap, poorly edited” signal strong disapproval. Comparisons to *Babylon 5* and *Dallas* (both framed negatively here) amplify disdain.  
2. **Rating Justification (2/10):** While the reviewer acknowledges their bias toward sci-fi, the sheer volume of flaws—technical, narrative, and performative—warrants an extremely low score. The show’s failure to meet even basic genre standards (e.g., creativity, character engagement) suggests it has almost no redeeming qualities.  
3. **Key Criticisms:**  
   - **Production Quality:** Mocked for “silly prosthetics” and CG that “doesn’t match the background,” indicating amateurish execution.  
   - **Characters/Dialogue:** Described as “wooden,” “predictable,” and devoid of a “spark of life,” undermining emotional investment.  
   - **Editing/Integrity:** The critique of abrupt actor replacements and reliance on Roddenberry’s name implies a lack of originality and effort to retain viewers.  
4. **Final Judgment:** The review concludes that the show’s flaws are irredeemable, even for forgiving sci-fi fans. The reference to Roddenberry’s ashes “turning in their orbit” seals the verdict: this is a creatively bankrupt, poorly executed project.

## Compare Reasoning vs Fast Model

See how a reasoning model (DeepSeek-R1) compares to a fast model (GPT-4o-mini)
on the same review.

In [9]:
def compare_models_on_review(review_text, ground_truth):
    """Compare reasoning vs fast model on the same review."""
    truth_label = "POSITIVE" if ground_truth == 1 else "NEGATIVE"
    
    display(Markdown(f"**Review (preview):** {review_text[:200]}..."))
    display(Markdown(f"**Ground Truth:** {truth_label}\n\n---"))
    
    for name, model_id in [("DeepSeek R1 (Reasoning)", REASONING_MODEL), ("GPT-4o-mini (Fast)", FAST_MODEL)]:
        print(f"\nAsking {name}...")
        result = analyze_review(review_text, model=model_id)
        display(Markdown(f"### {name}\n{result}\n\n---"))

In [10]:
# Pick a review and compare both models
sample = dataset[42]  # Change index to try different reviews
compare_models_on_review(sample["text"], sample["label"])

**Review (preview):** I have no idea how anyone can give this movie high marks. I didn't rent it thinking it was the next great horror flick, the next great horror spoof, or the next great low-budget horror spoof. Obviousl...

**Ground Truth:** NEGATIVE

---


Asking DeepSeek R1 (Reasoning)...


### DeepSeek R1 (Reasoning)
**Sentiment:** Negative  
**Rating:** 2/10  

**Key Points:**  
- **Severe sound editing issues** made the viewing experience frustrating, requiring constant volume adjustments.  
- **Technical flaws overshadowed any potential merit** the film might have had as a low-budget spoof.  
- **Distraction from engagement** due to the need to anticipate abrupt volume changes.  

**Reasoning:**  
1. **Sentiment Analysis**: The review is overwhelmingly negative. While the reviewer acknowledges the film might have been a "decent example" of its genre, the frustration with technical flaws dominates the tone.  
2. **Rating Justification**:  
   - The sound editing is described as "horrible," with extreme volume fluctuations (dialogue at 2, screams at 9), which the reviewer deems a "fatal flaw." This alone justifies a very low score.  
   - The frustration of constant volume adjustments ("toggling between 2 and 9") is emphasized as a dealbreaker, making enjoyment impossible.  
   - Despite potential merits (e.g., genre execution), the technical issues render the film unwatchable. A 2/10 reflects that even minor positives are negated by a critical weakness.  
3. **Key Points**: The focus on sound issues as a distraction and the admission that the flaws ruined the experience highlight why the reviewer’s verdict is harsh. The absence of praise for acting, humor, or plot further solidifies the negative assessment.

---


Asking GPT-4o-mini (Fast)...


### GPT-4o-mini (Fast)
**Sentiment:** Negative  
**Rating:** 3/10  

**Key Points:**
- The sound editing was described as "horrible," significantly detracting from the viewing experience.
- The reviewer struggled with volume control throughout the movie, making it difficult to engage with the content.
- Despite acknowledging that the movie might have some merit, the technical flaws overshadowed any potential enjoyment.

**Reasoning:**
1. The review opens with a strong negative sentiment, indicating that the reviewer is perplexed by the positive ratings from others. This sets a negative tone right from the start.
2. The primary criticism revolves around the sound editing, which the reviewer describes in detail as problematic. They mention having to constantly adjust the volume, which is a significant distraction and detracts from the overall experience.
3. The reviewer acknowledges that the movie might have some redeeming qualities but emphasizes that the sound issues prevent any enjoyment. This suggests that while there may be some merit to the film, the overwhelming negative experience caused by the sound editing leads to a low rating.
4. Based on the review's overall tone and the specific criticisms mentioned, a rating of 3/10 reflects the severe impact of the technical flaws on the viewer's experience, while still recognizing that there might be some potential in the film that was ultimately overshadowed.

---

## Batch Accuracy Test

Test how well the model classifies sentiment compared to ground truth labels.

In [11]:
def batch_accuracy_test(n_samples=10, model=FAST_MODEL):
    """Test model accuracy on a small batch of reviews."""
    import random
    random.seed(42)
    indices = random.sample(range(len(dataset)), n_samples)
    
    correct = 0
    results = []
    
    for i, idx in enumerate(indices):
        review = dataset[idx]["text"]
        label = dataset[idx]["label"]
        truth = "Positive" if label == 1 else "Negative"
        
        print(f"[{i+1}/{n_samples}] Analyzing...")
        output = analyze_review(review, model=model)
        
        # Simple check: does the output contain the right sentiment?
        predicted_positive = "positive" in output.lower().split("sentiment")[0:2].__str__().lower() if "sentiment" in output.lower() else "positive" in output[:200].lower()
        predicted = "Positive" if predicted_positive else "Negative"
        match = predicted == truth
        if match:
            correct += 1
        
        results.append({"truth": truth, "predicted": predicted, "match": match})
    
    accuracy = correct / n_samples * 100
    
    lines = [
        f"## Accuracy Test Results ({model})",
        f"**Samples:** {n_samples} | **Correct:** {correct} | **Accuracy:** {accuracy:.0f}%\n",
        "| # | Ground Truth | Predicted | Match |",
        "|---|-------------|-----------|-------|"
    ]
    for i, r in enumerate(results, 1):
        icon = "Yes" if r["match"] else "No"
        lines.append(f"| {i} | {r['truth']} | {r['predicted']} | {icon} |")
    
    display(Markdown("\n".join(lines)))
    return accuracy

In [12]:
# Run accuracy test with 10 samples (increase for more reliable results)
accuracy = batch_accuracy_test(n_samples=10, model=FAST_MODEL)

[1/10] Analyzing...
[2/10] Analyzing...
[3/10] Analyzing...
[4/10] Analyzing...
[5/10] Analyzing...
[6/10] Analyzing...
[7/10] Analyzing...
[8/10] Analyzing...
[9/10] Analyzing...
[10/10] Analyzing...


## Accuracy Test Results (openai/gpt-4o-mini)
**Samples:** 10 | **Correct:** 7 | **Accuracy:** 70%

| # | Ground Truth | Predicted | Match |
|---|-------------|-----------|-------|
| 1 | Positive | Positive | Yes |
| 2 | Negative | Negative | Yes |
| 3 | Negative | Positive | No |
| 4 | Positive | Positive | Yes |
| 5 | Negative | Negative | Yes |
| 6 | Negative | Negative | Yes |
| 7 | Negative | Positive | No |
| 8 | Negative | Negative | Yes |
| 9 | Positive | Positive | Yes |
| 10 | Negative | Positive | No |

## Gradio Interface

Interactive UI: paste any movie review or pick a random one from the dataset.

In [13]:
def analyze_ui(review_text, model_choice):
    """Gradio handler for review analysis."""
    if not review_text.strip():
        return "Please enter a movie review."
    
    model_map = {
        "DeepSeek R1 (Reasoning)": REASONING_MODEL,
        "GPT-4o-mini (Fast)": FAST_MODEL
    }
    model = model_map.get(model_choice, FAST_MODEL)
    return analyze_review(review_text, model=model)


def random_review():
    """Pick a random review from the dataset."""
    import random
    idx = random.randint(0, len(dataset) - 1)
    review = dataset[idx]["text"]
    label = "POSITIVE" if dataset[idx]["label"] == 1 else "NEGATIVE"
    return f"[Ground truth: {label}]\n\n{review}"


with gr.Blocks(title="Movie Review Rater") as demo:
    gr.Markdown("# Movie Review Rater\nPaste a movie review or load a random one from IMDB dataset.")
    
    with gr.Row():
        with gr.Column():
            review_input = gr.Textbox(lines=8, placeholder="Paste a movie review...", label="Movie Review")
            model_choice = gr.Radio(
                choices=["DeepSeek R1 (Reasoning)", "GPT-4o-mini (Fast)"],
                value="GPT-4o-mini (Fast)",
                label="Model"
            )
            with gr.Row():
                random_btn = gr.Button("Random IMDB Review")
                analyze_btn = gr.Button("Analyze", variant="primary")
        
        with gr.Column():
            output = gr.Markdown(label="Analysis")
    
    random_btn.click(fn=random_review, outputs=review_input)
    analyze_btn.click(fn=analyze_ui, inputs=[review_input, model_choice], outputs=output)

print("Launching Movie Review Rater...")
demo.launch()

Launching Movie Review Rater...
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Key Takeaways

1. **HuggingFace `datasets`** makes it easy to load and explore thousands of ML datasets
2. **Reasoning models** (like DeepSeek-R1) show their thinking process, making outputs more trustworthy
3. **Benchmarking on real data** (IMDB labels) lets us measure actual model accuracy
4. **Model choice matters** - reasoning models give better explanations but are slower; fast models are good for batch tasks
5. **Gradio Blocks** gives more control over UI layout than `gr.Interface`